In [ ]:
import base64, hashlib, importlib.util, os, subprocess, sys, threading
from getpass import getpass
from pathlib import Path

# Self-contained Colab Bridge Agent wheel (no GitHub dependency).
_WHEEL_NAME = "colab_bridge_agent-0.1.0-py3-none-any.whl"
_WHEEL_SHA256 = "5432f4440f89f8d0cf5b0c5c7390882277fc7d1ff0ca64bb9aca82293ba13448"
_WHEEL_B64 = """UEsDBBQAAAAIANxTLF2j8ObcGAAAABYAAAAeAAAAY29sYWJfYnJpZGdlX2FnZW50L19faW5pdF9fLnB5i48vSy0qzszPi49XsFVQMtAz1DNQ4gIAUEsDBBQAAAAIABxULF3DFhGyDwMAAPIJAAAcAAAAY29sYWJfYnJpZGdlX2FnZW50L2NsaWVudC5webVVbWvbMBD+7l8h/MkujvvCNkYghVI22AtdYWUMxjCKfU7VOpKQ5NHQ5b/vJDtW5CQtHVQfYunuOd09d6dLrcSSFEXdmlZBURC2lEIZQjkXhhomuI6iXnanBY9qizcryfhig73gqwFza4x8iKKobKjW5JquGkGrGyG+UrWA5AdtWviglFDpNCK4JKIG9MUCuLlsGP522gpqDI1xZooicRK7NDR1NpyoNSpa1UyJNmosv4fVSH7ktzbWouz8dYHnnXfyl1wJDmTmPt5gSR8K2VEq5isDekoYomfk7O07ckROT87eeLBhSxCtKTSUglcIrdHOgk9P8pMOlpLJuXMxDcjlAydED/tcIQ8mk/g4TvfBkeoAx30IsexQu8WYCBVwTkaX7nBF8x1ZaDJijAYjSeSrKoU2iaskmYsKi1Sx0vyylbLt9NtlJhT5HAEvRQUV3m87Mq/apdSJvSXDOCRV1AilZ0mcxRmJp3GaoYW23U11ydjsI200pHl3SxK3pp6830opq0kDPOmdpOT8QD58PHYpyjTs9Hsd9yYEHkoAzMnj/svWxH22wlCgJT4+24RDBXOXtcBv2C9ZoMOcG1TMeiqh8hZoBZimx0BqV3zZ2U1uVhLiKYmplA0r3TA4thmPs12bn5NL0dD5xL3hyRdYoWHYnKHROjz2jTLb10ceuZud3OW9qIUqNI6rVm+18R87bDB9A9bGngSFxhlHmGYcbXkJibPIXN+l+8rrx1cSO17k4voT3o+jk2M/csEnYn4HpSGfv3+7CoppIV1E/hFgCZSZA908BNVyS71gVTezSN8iL3ocvSuXyO6VPcZC2jIO7uyz8L5Q5Q+o6Z2iuN+tUx+y5lTqW3Ew4nvGXyf4jeOnY7fuUWY/zzFRsGDagDrEBJsZmlehsvH8NBXnH4Xu+xwZKZpmP5H/jNFe+ER8YSJ125hDf8/jvA6KUiyXlFe7iu4dj4QHKtD/UXsc2MdZ2HHnLgj0L01FMACG2lmyowF4sIohyjNGlD+MUB19Oz3dZqTd6YKR3vNHiD94FFbuH1BLAwQUAAAACAAKVCxd6qU0qjMBAABkAgAAHAAAAGNvbGFiX2JyaWRnZV9hZ2VudC9jb25maWcucHmFkFFrwjAUhd/zK0KeWlAnPgodVg1uTBS0exhDQmxvNaxNJLkdbL9+ad266saWp+Tj3HNPTm5NSYXIK6wsCEFVeTIWqdTaoERltCMkrzWZRJkW0jlwX6IWkU9gvJhMWhz4wXfQUWIrCEmDaHwAjTOjc3UYE+qPrIGobDGmDm0HvcDbNyrkHs4KGlGWGv/u20qjKoE1giNIi3uQKBykRmduTJVGLx4NSSOYNPtLwKPJGpBBTuufCdCvQVq4kPZvKevkY+eA9bHg69HUq4KWXYSPjBt4H2WNfmaz9TKeiunmfr7gIl7wVSIeN0u2G1ifX50CdsPC3i8+/sf/+TzwJ7a7nG2q6cwNDoDB5ay/8iXrXRd3FeJHhZFvMPjT+I7Hm2TK40Rs+Wy9mm/rJaMhCzvWIfkAUEsDBBQAAAAIABxULF2RJgXSjgMAAPMLAAAaAAAAY29sYWJfYnJpZGdlX2FnZW50L21haW4ucHmtVltP2zAUfs+vsPLUsKzapD1VKlIHfZhWirTB0FRVkUlPikeaRLYDVIz/vuNLajstMKbloU18Ph+f63dc8HpDsqxoZcshywjbNDWXhFZVLalkdSWiqFAYuW1Yte7kJ7Qs6XUJURRNZrPzq+lpdnJ+djaZn34nY/IYcyg4iJts3bRxSnafvK0k24C/1PA6ByFAxE+obAUFgQfIWwlZXm82tFoN7P+IrFguU3KUEtxzDcIsLITk6c6exWKZ6uXlMiHvj/XrKCL4WC0ZW6GBuKdTO1yDHMRspWyKkyTAos+A6ADpy2IDZ0W4A0NHWEX2AlPzgzjrjValHg6Yi4o87hbUEzv745HnTBqiBCatFYjA+P6CXAL6FSKA85pjbFegUJfzr/Pzq3lnYx/c0G1ZU3XivK7ACZ/0m+RbZ7WFYriMPwvf1eXARAoecmgkmeo/LK7/7XRBWfmayxfT2fRsevHtZ3Y5n/yYfJlNPs+mb3N8z9jXDfWMRHFTQpia0MbwTM8Y+2ZkXb9gU2V0DZUc2OKtCrY2kCPzl5cMxWOnVsi6yeAuXNzQhyzf5qXqLFZJ8lvbgQl1kL9qvP5G3YgINsm2BKJowP9uW7aK9IJmm6ExuZNOlHsnesnDGGs6TF6XJVa8opxMVLQRNzWyRbdqaeaAxHLSTmKsoLlkd8hBxoyxDaFqYc+WgYn1UEc/a3mZkmDlFraJr80a3LWI0ubVkM+Zo8Pu7IM7Rh0968+BTY5zR88HKPLq3arccafK1lD9fBpYyuwgjgYCnxd7BneUEAR6yGHNhAQ+cEfuYop1Bhjh3kmJyZcpXDz2g/68v0EiIBe8BUcxSNOu7gkTmn91pWKzeqIhwygg1SejgBOuOdDbyBHWvzqsHpXUVzeqOvA37bL2+laXYF9BGOou0UGoYzeh9+L8FjVm7ntuvm27sz/dd/s5VTdAubwGanmwn6j+2AhHje4PmudQAqey5tgZnvVm9PtiNLKtbqv6voqTdF+TROAGcEBm9A5nkmLIQxoPwVJdtj2lT+4zcTXYYO9mmPAGr2ngqsHGQ0m9sLqwFe4iou8fvpbgkqPCv1j2+qBuJYrVcc9c1Lr72TgoziRQ0u960Za9tL2UOs+MhT96ly/A7Px9CdLN2Zcw3pzuwby8WDJ6NyYfffZxA3aPfezy8difwgfop/tSYRmKEqDpZtCu+pG7cGklrD32ruJCGf0BUEsDBBQAAAAIANxTLF3aemqpKwEAAFkCAAAcAAAAY29sYWJfYnJpZGdlX2FnZW50L21vZGVscy5weXVR227DIAx95yssnsiWRpEm7SFa9wn7gL0glpgNiUAFzu7795mmtJW2IgEyPgcfH9sUZ9DaLrQk1BrcvIuJwIQQyZCLIQthC2YyhORmrIgat1DOzxhQCDGhhdGbnJ390GkJJaXNMyreOuMYw5QHsD4aamBzD5nSIICXs3AGgbst3PZrpqyErC6A9O4V5QX8Tf8PIZPxB0Z9itZ6F/hxVVtFMpKWrEur+gVNoic0pLgVOoVD0dvCVQshvg0nR77hgduH7f7a9zW5kVY145ISBuIkcyCmI6vjWFXruoXGZo8/FmPGEVpUuRxtTPMfUV3CnTcjKvkoW5DXbEPfy2b9jT3if2bzrvqub0FVNZtTnaYjnrSvVqoD82DXl1yNkcPFuTZc9WwWjOToR/wCUEsDBBQAAAAIAHRVLF1UF0nWxQcAAG8eAAAcAAAAY29sYWJfYnJpZGdlX2FnZW50L3Byb2Jlcy5wed1YbW/cNgz+fr/C8L7Ync9bu7cu2G3IumtRrEuLNi0weAdDsXWJNp/kyXJelt1/HylZtuTzXVLsBcUOSGxLpEhRJB+Kayk2QZ6vW9VKmucB29RCqoBwLhRRTPBmNuvGiubSvjJh3yS1b017VktR0KaZrXFRdVMzfm4XfEKqipxVdDablXQd5EVFCY8uSdXSo6BRMg7m3+LzaBbAT1JQhwd6OoVRVkex5WRcjflgKPgzOBGcGvbLYOFJiPUoW8ME48FtGCZBePLJMT6yE6GCN22NStJyFW7NCo4SuKyrFMpfV4KAFnGvVDfgq6UH/zvFrFJWp7ZlZd606zW73mtoPb5Xqc78RKrmiqmLKHz26u08jAdNLLt+Zp8frfRMjQx2NG3qiqkoBD67bkV5pGni4NtF8GhnY+vwVk9n80erbA6rbud24OFqG+54SDZ/DJLNroGuofl53ebgrpGi10rvOgkeJEHRliS/pLIBt9aj3dmAqvjQlqlYo7KSFWrVeaK4ao6cUaDNzC7XQuIsHhyISiUlJZURE+kb8Fd+/vyllh471oKtQ1RpJuCFgIjws6BVZX1cL4oDuCrQOcz4KwRXjLd0MBhol5K6pryMPMpb7wt/IeMlvQ6PTAABY/bpKk52yTjZUKQyDoF0DyfpHO9CctfZkOnRJNOGboS8yRXklirfsDNXnc8OcbQNLccMnx9iWEtKxwxfTG9EsYr9oXOd9puayoJy5TJ+Ocmo6AZoiU6chUv+1SR5La6ozEtJrvIrpDbhivSPD9BXbMPUmOHrSYZSMnBu6+CjM5w+bDcigMH99Km3/Vfshh/6nxd4HQL4wfc/Dyx0mnsGV81Kl2g6sqwNdyNxOqhQfuf1d4XU3ceY80tWMpK7rhBB7pQt51Qe9TgOxzUAfgqzFlc8vFPyZjB0ITZ1RQHMgNcs55s2C43sebNh4cpXvCC1jjPRqrpVi1PZUp8AXWJiuLigxW9T5GxDYa3FF8nIIvS6oLUKoqesooDAT0XLy6WUAhDE2S9agZavzNfu9KlZfXldM0nLeD9yb4gqLtAcAJOUyOIikuGTtz8cB+86lPqleRBln86/Xn0cfXf0S2pe4wcxVAi9OcHHS5DmnaheOD2Xoq2jhzEGiBFFq4Ya8d1xKwFSc+NCipREkUifJMbkxBl21ZzmCoKPsMiDsoKdcyGpa7+lfuAOxnv3Ayisb4wG2t/IJWHau8CJnxJQNTlAPKQt3M+IEo3TKsjLpCZnkN8Vow1Q3m4HOhMLvUg4hTMhqkhLSFFCyppBoyg29nUXPNJWyiDaEvR8TGG3W1vk9JyDBTxL6sXAuxRwYbw6ckt6yQrQHWetXPvDPKaTjU5khJ/TSNONEppxrl/RLzcMDgeEOALOqcqtELufm0gvG+8s4+440zS4UajR9Prb9FYL6Mqywy4wXm4w2I5z7HeM/j3ZQzw4BuyTKCUj+9REUFtbisTUfYkBwv5zWHePG7mfhnrbxVMhIDEUSgdUw0ndXAhlstwDQ3hnHjVkZjswfEYH2ixbJYEFy4nITWaj0P29pfIGaIc8a6rABKElwZItMcCR6orMfmCxZd+xjkqcEikFgclw1m4RpKd01ZJildO96gImGdUmmj/+mxCRgBPO53qPcxC9uNWvW7w8zecQJ5DxFlBAJFxc6BICXlrOVPP/xpZRfiUFVD14QEKC44ZF3YajVAmmQ5/ORlaBk60oeBZUFXelZYpKQroqkSA8eZnDNXEs5Z/P8xBy+hQcGgyLqfrFuNLCPCbczkjDMLJ50oRe5J3bRELzGD8wZMNzBa38C/G4aPAvxQv3YxfveuPghhFEomltEtCmv+yzhvFGEV4AULlpU6cpx30R2EBLhDXU3AcNg3gLnDFyTb3tgxVMZrsK3YSYLV3RegUDdiMFTXJEOI7dSgkX3wWoUWR1fgdKhLiqtr5eYxRzNt7w4YxOR5uuSJDSBSUv3HT/wpe3E4D7XXGoeJwz3UOdGMeN47tx9+CKliyeRk57++nR896Xjz0lq7meXcKVXmgwvAeueLBinALB8sNFFs30ftjZ77GLmDncdpsFGkvXBTX+ORfRRHdghqvmh2uL/wZl7S2ddth5GC/HGHm6fLH8aXn6+uf87cnxu+PnL46/f7EMO0TTrTQlTE+hu2PoFibkJeeOcXebxPN8e0v02ybYisV2SPCN24c17jRqidgc7LY4+knUeWgjd70NV1B3Z2l0o0YnLcJLzeaL9Taf4ddKX5Dgy1int/v7NIx27sr/avPIBpFvENdc/SZsj2lUO7gtJc8iOp/a9eNxEXFHd+l9Okv37SptvbaDHxb9+97o0GE9gWhjUIB8hvE+gMIo4XdtCdG4X3VFFCYodwxXsYDetJXyyjbEqQvBHSCzS6T+TORhYEfjkXcvHuFvVHJauWQSjEIa6lFtSHHBOHXJuiGPjDVgLzCjBXHRpBTyu8QbGkLuk5eQU/LXkGeO3yzz0+NnYYzuPU32/fGTH5cnP+Tvlq/fPH95Esb+HRjE5J2quSLnIPIe0lwLdW7X1uYQKURMiQ6Cn+lGQLwJzoqorwjw/z/XdsJzTtsabsg0uu1rkVGRfWe9frBG76IA66+9kqcL/t1mhW1S5HYuz3c7E94aB8q6yUbW/e4ef7d54mcGY4zZX1BLAwQUAAAACADlVSxd5H9O43UCAAApBAAAMwAAAGNvbGFiX2JyaWRnZV9hZ2VudC0wLjEuMC5kaXN0LWluZm8vbGljZW5zZXMvTElDRU5TRV1SzW7bMAy+6ymInFrA6IYedthNiZVGm215srIuR8dWYg2OFVjygr79SCdt1wEBDJH8/sjk0kDmGjsEy9jKn19Gd+wi3DX38Pj58Qv8mL65oa0dY6UdTy4E5wdwATo72v0LHMd6iLZN4DBaC/4ATVePR5tA9FAPL3C2Y0CA38faDW44Qg0NijCcjB3SBH+Il3q0ONxCHYJvXI180PpmOtkh1pH0Dq63Ae5iZ2FR3RCL+1mktXXP3ADUe23BxcXOTxFGG+LoGuJIwA1NP7Xk4bXdu5O7KRB8Th4Ykk4BE5DPBE6+dQf62jnWedr3LnQJtI6o91PEYqDivMKEcnzyIwTb9wwZHPqes767m2fI+pkWGm8rClS5dP70MYkL7DCNA0raGdN6XNms+Ns2kSo0fvB97y8UrfFD6yhR+MqYwVa993/snOV62MFHtHq1QAc4v1/11gpd3fewt7eFoS6ut/4nzkjyIeLhXd3D2Y+z3v8xH1B/I6BSa/PMtQBZQanVT5mKFBa8wvcigWdpNmprACc0L8wO1Bp4sYPvskgTEL9KLaoKlGYyLzMpsCaLVbZNZfEES8QVCv+9MpcGSY0CErxRSVERWS70aoNPvpSZNLuEraUpiHOtNHAouTZytc24hnKrS1UJlE+RtpDFWqOKyEVhHlAVayB+4gOqDc8ykmJ8i+41+YOVKndaPm0MbFSWCiwuBTrjy0xcpTDUKuMyTyDlOX8SM0ohi2Y0dnUHzxtBJdLj+FsZqQqKsVKF0fhMMKU2b9BnWYkEuJYVLWStVZ4wWici1EyCuEJcWWjV8OEiOELvbSXeCCEVPEOuisAU8XX4gf0FUEsDBBQAAAAIAOVVLF3KfBB7tgAAAPYAAAArAAAAY29sYWJfYnJpZGdlX2FnZW50LTAuMS4wLmRpc3QtaW5mby9NRVRBREFUQUXNywrCMBCF4X2eYh7AhFYFIagLrwgqouJ+bMcayEXTKZi3t4qX/fefsyHGEhnliWJtgtfQVX2xRUcaimDxLM/RlBVJrMiz+KlM5SoTh8Y5jEnDnrCUwdsEyxAqSzB9tRAbz8YRMFlyxDHBewYuIX7E5L0u1qYgX7efm9VR7OnemEi13CW+vs7Go57Ksy+SC2NbuV5N59vD/K9npmYNV+bbY5h3xqNMdQdiljw6U2iwn/jSxuIJUEsDBBQAAAAIAOdVLF0nTOaKXAAAAFsAAAAoAAAAY29sYWJfYnJpZGdlX2FnZW50LTAuMS4wLmRpc3QtaW5mby9XSEVFTAXBMQrDMAwF0F2n0NgOMkm6BF+gdCslJLMLnyZgpCDLQ27f97YdqLLC22GaeUwDPaHwEuaZG6KfYVYb3+YpDWm808cs5NXk3R31+GYO76Cl/DKf10PUFFL0IvoDUEsDBBQAAAAIAOVVLF3idcfEFQAAABMAAAAwAAAAY29sYWJfYnJpZGdlX2FnZW50LTAuMS4wLmRpc3QtaW5mby90b3BfbGV2ZWwudHh0S87PSUyKTyrKTElPjU9MT80r4QIAUEsDBBQAAAAIAOdVLF3SQqjfHAIAALsDAAApAAAAY29sYWJfYnJpZGdlX2FnZW50LTAuMS4wLmRpc3QtaW5mby9SRUNPUkSN0styqkAQgOF9ngUNg3JxkQU3AYFAkPuG4u44hEEHJfL052yssiossurd93d1dYm7vMiKK6zaOsvbuh/fswz2cMyy9fCgyClnWO4DBR7rifuS1+UaXqou3nBV6ugoMNjJ4MJsu/EGtVSEYkVTDPNW/kbLDv4fL+ThniXczcJD55Fo3/EklVVefyByYJrqem7OSaTJeWNb34Ri2C23iOK+ge0LWsQDbE0aXj2+wn4pEpFhxdS8KFWT+JthRDbrz59MbaklxYHFRb9z2L+IoD4an6IB6AvkJnLe5xfN9xIaIDzydphmtinJVntrJBtRG5rdLZK4qjvygnJEn61KDY0fo+1SB0X7r/sXM9tGSzBnZmAzXJDl3CdXMyiOBkvmcMVF/Wq6+EqIeY8C71FgkquMIKf34bifAdPDvpcE+1wqvEb3wZbi+d0SuqLXYE2vK0jGFewb/N7Bsu5JTd4tQ1Y/j+qzNZ+OM9kiFksgbPxjkrkHlwHeqZvm2aGVINFCgfenR5yIFKA59i8tW/VFRfTFZ2M671yWjWKs3TyVvbiW5zdSWpajYXYOCsYK7J1DrteOFVDM4nv8SkS6qlpPP68TCFsadImmzFJ7FPSfKHbazS3kgLaDZ4fchhXaYUAQ9bdrjXjIuvped+vxZ3xmwtiR8q8dqjjh4d48jk5b00E7W4iL6SANJxx6NREka0A0BZa+51fGU2XHUyjq7R9QSwECFAMUAAAACADcUyxdo/Dm3BgAAAAWAAAAHgAAAAAAAAAAAAAApIEAAAAAY29sYWJfYnJpZGdlX2FnZW50L19faW5pdF9fLnB5UEsBAhQDFAAAAAgAHFQsXcMWEbIPAwAA8gkAABwAAAAAAAAAAAAAAKSBVAAAAGNvbGFiX2JyaWRnZV9hZ2VudC9jbGllbnQucHlQSwECFAMUAAAACAAKVCxd6qU0qjMBAABkAgAAHAAAAAAAAAAAAAAApIGdAwAAY29sYWJfYnJpZGdlX2FnZW50L2NvbmZpZy5weVBLAQIUAxQAAAAIABxULF2RJgXSjgMAAPMLAAAaAAAAAAAAAAAAAACkgQoFAABjb2xhYl9icmlkZ2VfYWdlbnQvbWFpbi5weVBLAQIUAxQAAAAIANxTLF3aemqpKwEAAFkCAAAcAAAAAAAAAAAAAACkgdAIAABjb2xhYl9icmlkZ2VfYWdlbnQvbW9kZWxzLnB5UEsBAhQDFAAAAAgAdFUsXVQXSdbFBwAAbx4AABwAAAAAAAAAAAAAAKSBNQoAAGNvbGFiX2JyaWRnZV9hZ2VudC9wcm9iZXMucHlQSwECFAMUAAAACADlVSxd5H9O43UCAAApBAAAMwAAAAAAAAAAAAAApIE0EgAAY29sYWJfYnJpZGdlX2FnZW50LTAuMS4wLmRpc3QtaW5mby9saWNlbnNlcy9MSUNFTlNFUEsBAhQDFAAAAAgA5VUsXcp8EHu2AAAA9gAAACsAAAAAAAAAAAAAAKSB+hQAAGNvbGFiX2JyaWRnZV9hZ2VudC0wLjEuMC5kaXN0LWluZm8vTUVUQURBVEFQSwECFAMUAAAACADnVSxdJ0zmilwAAABbAAAAKAAAAAAAAAAAAAAApIH5FQAAY29sYWJfYnJpZGdlX2FnZW50LTAuMS4wLmRpc3QtaW5mby9XSEVFTFBLAQIUAxQAAAAIAOVVLF3idcfEFQAAABMAAAAwAAAAAAAAAAAAAACkgZsWAABjb2xhYl9icmlkZ2VfYWdlbnQtMC4xLjAuZGlzdC1pbmZvL3RvcF9sZXZlbC50eHRQSwECFAMUAAAACADnVSxd0kKo3xwCAAC7AwAAKQAAAAAAAAAAAAAAtIH+FgAAY29sYWJfYnJpZGdlX2FnZW50LTAuMS4wLmRpc3QtaW5mby9SRUNPUkRQSwUGAAAAAAsACwCBAwAAYRkAAAAA"""

if importlib.util.find_spec("colab_bridge_agent") is None:
    wheel_bytes = base64.b64decode(_WHEEL_B64)
    if hashlib.sha256(wheel_bytes).hexdigest() != _WHEEL_SHA256:
        raise RuntimeError("Embedded Colab Bridge wheel integrity check failed")
    wheel_path = Path("/tmp") / _WHEEL_NAME
    wheel_path.write_bytes(wheel_bytes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", str(wheel_path)])

agent_url = input("Colab Bridge Agent URL: ").strip()
agent_key = getpass("Colab Bridge Agent key: ").strip()
label = input("Runtime label [colab-runtime]: ").strip() or "colab-runtime"
if not agent_url or not agent_key:
    raise ValueError("Agent URL and key are required")

os.environ["COLAB_BRIDGE_AGENT_URL"] = agent_url
os.environ["COLAB_BRIDGE_AGENT_KEY"] = agent_key
os.environ["COLAB_BRIDGE_LABEL"] = label

from colab_bridge_agent.config import AgentConfig
from colab_bridge_agent.main import run_agent
from colab_bridge_agent.probes import collect_gpu_snapshot

if globals().get("_COLAB_BRIDGE_THREAD") and _COLAB_BRIDGE_THREAD.is_alive():
    print("Colab Bridge Agent is already running.")
else:
    _COLAB_BRIDGE_THREAD = threading.Thread(
        target=run_agent,
        args=(AgentConfig.from_env(),),
        name="colab-bridge-agent",
        daemon=True,
    )
    _COLAB_BRIDGE_THREAD.start()
    print("Colab Bridge Agent started in the background.")

print("Local accelerator check:", collect_gpu_snapshot())
